# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas  
**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI).

**Nota de elaboración:** Las celdas Markdown cuentan con apoyo de inteligencia artificial generativa. Los ejemplos tienen fines educativos y demostrativos. El autor es responsable de su revisión, adaptación y validación en Microsoft Fabric.

# NB_01 — Clasificación de textos con `ai.classify`

**Serie:** Microsoft Fabric AI Functions con PySpark  
**Modelo de referencia:** `gpt-5-mini`  
**Entorno:** Microsoft Fabric Notebook — PySpark  
**Tipo de endpoint:** endpoint LLM integrado de Fabric

## Objetivo

Aprender a utilizar `ai.classify()` para asignar una categoría a cada registro de texto de un DataFrame de Spark, utilizando un conjunto de etiquetas definido por el usuario.

En este laboratorio también revisaremos las estadísticas de ejecución (`ai.stats`), los tokens reportados y una estimación ilustrativa del consumo de capacidad.

> **Importante:** las respuestas generadas por IA pueden contener errores. Revise los resultados antes de utilizarlos en procesos productivos.

## Documentación oficial

- [AI Functions — descripción general](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/overview)
- [Usar `ai.classify` con PySpark](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/classify)
- [Personalizar AI Functions con PySpark](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration)
- [Facturación de AI Functions](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)


## 1. Requisitos y consideraciones

Antes de ejecutar este notebook, verifica:

- El workspace está asignado a una capacidad Fabric compatible (F2 o superior, o una capacidad P).
- El administrador habilitó la configuración de tenant requerida para Copilot y otras características basadas en Azure OpenAI.
- El notebook utiliza Fabric Runtime 1.3 o posterior.
- La sesión corresponde a un entorno PySpark de Microsoft Fabric.

La función `ai.classify()` procesa el texto de una columna y devuelve una etiqueta seleccionada entre las categorías proporcionadas.

**Nota de facturación:** el endpoint LLM integrado de Fabric genera consumo en el medidor **Copilot y AI**. El procesamiento Spark del notebook se mide separadamente en el medidor Spark. Si se configura un endpoint personalizado, el proveedor del modelo factura las llamadas y Fabric no cobra esas llamadas de modelo como AI Functions del endpoint integrado.

## 2. Configurar el modelo

Se utiliza el mismo patrón de configuración del notebook de referencia `NB_AIFUNCTIONS.ipynb`.

El modelo predeterminado documentado para AI Functions en Python es `gpt-5-mini`, con `reasoning_effort` configurado en `low`, salvo que se personalice la configuración.

In [1]:
import synapse.ml.spark.aifunc as aifunc

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

from datetime import datetime, timezone
import uuid

MODEL_NAME = "gpt-5-mini"

aifunc.default_conf.set_deployment_name(MODEL_NAME)

print(f"Modelo configurado: {MODEL_NAME}")

StatementMeta(, 803543f0-f257-486a-92d6-527086aa77a9, 3, Finished, Available, Finished, False)

Modelo configurado: gpt-5-mini


## 3. Crear un conjunto de datos de prueba

Utilizaremos solicitudes ficticias de soporte tecnológico. Cada registro contiene una descripción que deberá clasificarse en una de estas categorías:

- `Infraestructura`
- `Bases de datos`
- `Seguridad`
- `Redes`
- `Aplicaciones`
- `Otros`

Las categorías deben ser suficientemente claras y cubrir los escenarios esperados. La calidad de la clasificación depende, entre otros factores, de la claridad del texto y de las etiquetas suministradas.

In [2]:
data = [
    (1, "El servidor virtual presenta alta utilización de memoria y CPU."),
    (2, "La consulta SQL demora varios minutos y bloquea otras transacciones."),
    (3, "Se detectaron intentos de acceso no autorizado a una cuenta."),
    (4, "Los usuarios pierden conectividad con la red inalámbrica de la oficina."),
    (5, "La aplicación de gestión muestra un error al guardar una solicitud."),
    (6, "Se requiere ampliar el almacenamiento del clúster de procesamiento."),
    (7, "Una tabla presenta registros duplicados después de la carga ETL."),
    (8, "El certificado TLS del portal web está próximo a vencer."),
    (9, "El servicio de DNS no resuelve el nombre del servidor interno."),
    (10, "Se solicita incorporar un nuevo campo al formulario de atención.")
]

df = spark.createDataFrame(
    data,
    ["id_solicitud", "descripcion"]
)

display(df)

StatementMeta(, 803543f0-f257-486a-92d6-527086aa77a9, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 314bf1e6-97c7-4267-84f0-96c3e845dbcd)

## 4. Ejecutar `ai.classify()`

La sintaxis PySpark documentada es:

```python
df.ai.classify(
    labels=["category1", "category2"],
    input_col="text",
    output_col="classification"
)
```

Parámetros utilizados:

- `labels`: conjunto de etiquetas permitidas para la clasificación.
- `input_col`: columna que contiene el texto que se analizará.
- `output_col`: nombre de la columna donde se almacenará la categoría.
- `error_col`: columna opcional para registrar errores de procesamiento por fila.

La función devuelve un nuevo DataFrame de Spark.

In [3]:
run_id = str(uuid.uuid4())

LABELS = [
    "Infraestructura",
    "Bases de datos",
    "Seguridad",
    "Redes",
    "Aplicaciones",
    "Otros"
]

resultado = df.ai.classify(
    labels=LABELS,
    input_col="descripcion",
    output_col="categoria",
    error_col="error_clasificacion"
)

display(resultado)

StatementMeta(, 803543f0-f257-486a-92d6-527086aa77a9, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d4dc6dbf-b7d2-4970-a83c-9638f1034ce8)

## 5. Revisar la calidad del resultado

La clasificación es una inferencia del modelo y no reemplaza una validación de negocio. Revisaremos los resultados y detectaremos filas que no hayan recibido categoría o que presenten errores.

En un escenario productivo, se recomienda definir reglas de validación, revisar una muestra de resultados y establecer un proceso de tratamiento para las clasificaciones incorrectas.

In [4]:
display(
    resultado.select(
        "id_solicitud",
        "descripcion",
        "categoria",
        "error_clasificacion"
    )
)

# Resumen de categorías generadas
display(
    resultado
    .groupBy("categoria")
    .count()
    .orderBy(F.desc("count"))
)

# Registros sin categoría o con error
display(
    resultado.filter(
        F.col("categoria").isNull()
        | F.col("error_clasificacion").isNotNull()
    )
)

StatementMeta(, 803543f0-f257-486a-92d6-527086aa77a9, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 156b52e6-1ba9-47b0-b83f-63cdc325c322)

SynapseWidget(Synapse.DataFrame, eb7b77a0-31fc-41c6-b6b5-b0bc202c3147)

SynapseWidget(Synapse.DataFrame, e18c45c1-ac88-4eda-b228-11b50b7c308e)

## 6. Consultar estadísticas con `ai.stats`

Después de ejecutar una AI Function, las estadísticas se consultan desde el DataFrame resultante mediante `resultado.ai.stats`.

Entre las métricas documentadas se encuentran:

| Campo | Descripción |
|---|---|
| `num_successful` | Filas procesadas correctamente. |
| `num_exceptions` | Filas con excepciones. |
| `num_unevaluated` | Filas no evaluadas. |
| `num_harmful` | Filas bloqueadas por filtros de seguridad, cuando se informa. |
| `input_tokens` | Tokens de entrada. |
| `cached_tokens` | Tokens de entrada en caché. |
| `output_tokens` | Tokens de salida. |
| `reasoning_tokens` | Tokens de razonamiento reportados. |
| `client_type` | Tipo de cliente o endpoint. |
| `model` | Modelo utilizado. |

La disponibilidad de campos puede variar según la operación y la respuesta de estadísticas. Por eso, el código recupera las métricas de forma defensiva.

In [5]:
stats_df = resultado.ai.stats

display(stats_df)

stats_rows = stats_df.collect()

if not stats_rows:
    raise RuntimeError(
        "AI Functions no devolvió estadísticas de ejecución."
    )

# El ejemplo de referencia devuelve una fila agregada por operación.
stats = stats_rows[0].asDict(recursive=True)

print("===== ESTADÍSTICAS AI FUNCTIONS =====")
for key, value in stats.items():
    print(f"{key}: {value}")

StatementMeta(, 803543f0-f257-486a-92d6-527086aa77a9, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 978ab38f-ed2b-4616-badb-cd58a1ba1c7a)

===== ESTADÍSTICAS AI FUNCTIONS =====
num_successful: 10
num_exceptions: 0
num_unevaluated: 0
cached_tokens: 0
input_tokens: 744
output_tokens: 804
reasoning_tokens: 448
client_type: fabric_llm_endpoint
model: gpt-5-mini


## 7. Estimar el consumo de capacidad

La siguiente estimación corresponde a las tarifas publicadas para `gpt-5-mini` en el endpoint integrado de Fabric:

| Tipo de token | CU-segundos por 1.000 tokens |
|---|---:|
| Entrada | 8,40 |
| Entrada en caché | 0,84 |
| Salida | 67,23 |

La fórmula separa los tokens de entrada normales de los tokens en caché:

$$
CU_s = \\frac{(I-C)\\times 8.40 + C\\times 0.84 + O\\times 67.23}{1000}
$$

Donde:

- \(I\): tokens de entrada.
- \(C\): tokens de entrada en caché.
- \(O\): tokens de salida.

**Alcance:** esta cifra es una estimación del consumo de las llamadas al modelo integrado. No representa el consumo total de Spark ni el consumo completo de la capacidad del workspace.

Los `reasoning_tokens` se muestran como métrica informativa. No se suman por separado a la fórmula para evitar duplicarlos si ya están incluidos en el contador de tokens de salida reportado por el servicio.

In [6]:
# Métricas reportadas por ai.stats
input_tokens = int(stats.get("input_tokens") or 0)
cached_tokens = int(stats.get("cached_tokens") or 0)
output_tokens = int(stats.get("output_tokens") or 0)
reasoning_tokens = int(stats.get("reasoning_tokens") or 0)

# Validación defensiva: caché no puede superar el total de entrada.
if cached_tokens > input_tokens:
    raise ValueError(
        "cached_tokens no puede ser mayor que input_tokens."
    )

# Tarifas publicadas para gpt-5-mini (endpoint integrado).
INPUT_RATE = 8.40
CACHED_RATE = 0.84
OUTPUT_RATE = 67.23

regular_input_tokens = input_tokens - cached_tokens

estimated_cu_seconds = (
    regular_input_tokens * INPUT_RATE
    + cached_tokens * CACHED_RATE
    + output_tokens * OUTPUT_RATE
) / 1000

print("===== ESTIMACIÓN DE CONSUMO =====")
print(f"Modelo reportado: {stats.get('model', MODEL_NAME)}")
print(f"Tokens de entrada: {input_tokens:,}")
print(f"Tokens en caché: {cached_tokens:,}")
print(f"Tokens de entrada normales: {regular_input_tokens:,}")
print(f"Tokens de salida: {output_tokens:,}")
print(f"Tokens de razonamiento (informativo): {reasoning_tokens:,}")
print(f"CU-segundos estimados: {estimated_cu_seconds:.6f}")

StatementMeta(, 803543f0-f257-486a-92d6-527086aa77a9, 8, Finished, Available, Finished, False)

===== ESTIMACIÓN DE CONSUMO =====
Modelo reportado: gpt-5-mini
Tokens de entrada: 744
Tokens en caché: 0
Tokens de entrada normales: 744
Tokens de salida: 804
Tokens de razonamiento (informativo): 448
CU-segundos estimados: 60.302520


## 8. Construir un registro de auditoría

Para facilitar la trazabilidad, consolidaremos el identificador de ejecución, la operación, el modelo, las filas procesadas y las métricas de tokens.

Este ejemplo crea un DataFrame de auditoría en memoria. La persistencia en una tabla Delta se deja como paso opcional para que el notebook pueda ejecutarse sin crear o modificar tablas del Lakehouse.

In [ ]:
def get_int_metric(name):
    value = stats.get(name)
    return int(value) if value is not None else 0

num_input_rows = df.count()
num_successful = get_int_metric("num_successful")
num_exceptions = get_int_metric("num_exceptions")
num_unevaluated = get_int_metric("num_unevaluated")
num_harmful = get_int_metric("num_harmful")

if num_exceptions > 0 or num_harmful > 0:
    execution_status = "PARTIAL"
elif num_successful == num_input_rows:
    execution_status = "SUCCESS"
else:
    execution_status = "INCOMPLETE"

audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("execution_timestamp_utc", StringType(), False),
    StructField("notebook_name", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("model_configured", StringType(), True),
    StructField("model_reported", StringType(), True),
    StructField("num_input_rows", LongType(), True),
    StructField("num_successful", LongType(), True),
    StructField("num_exceptions", LongType(), True),
    StructField("num_unevaluated", LongType(), True),
    StructField("num_harmful", LongType(), True),
    StructField("input_tokens", LongType(), True),
    StructField("cached_tokens", LongType(), True),
    StructField("output_tokens", LongType(), True),
    StructField("reasoning_tokens", LongType(), True),
    StructField("estimated_cu_seconds", DoubleType(), True),
    StructField("client_type", StringType(), True),
    StructField("execution_status", StringType(), True)
])

audit_record = [(
    run_id,
    datetime.now(timezone.utc).isoformat(),
    "NB_01_AI_CLASSIFY",
    "ai.classify",
    MODEL_NAME,
    str(stats.get("model") or MODEL_NAME),
    num_input_rows,
    num_successful,
    num_exceptions,
    num_unevaluated,
    num_harmful,
    input_tokens,
    cached_tokens,
    output_tokens,
    reasoning_tokens,
    float(estimated_cu_seconds),
    str(stats.get("client_type") or "unknown"),
    execution_status
)]

audit_df = spark.createDataFrame(
    audit_record,
    schema=audit_schema
)

display(audit_df)

## 9. Persistencia opcional en Lakehouse

Si deseas guardar el registro de auditoría en una tabla Delta del Lakehouse adjunto al notebook, ejecuta la siguiente celda.

**Atención:** el modo `append` agrega un registro por ejecución. Verifica el Lakehouse y el nombre de tabla antes de habilitar la escritura.

In [ ]:
# Descomenta para persistir la auditoría en el Lakehouse adjunto.

# TABLE_NAME = "dbo.audit_ai_functions"
#
# (
#     audit_df.write
#     .format("delta")
#     .mode("append")
#     .saveAsTable(TABLE_NAME)
# )
#
# print(f"Auditoría guardada: {TABLE_NAME}")

## 10. Conclusiones

En este laboratorio:

1. Configuramos el modelo de AI Functions.
2. Construimos un DataFrame PySpark con solicitudes ficticias.
3. Ejecutamos `ai.classify()` utilizando etiquetas personalizadas.
4. Revisamos los resultados y posibles errores.
5. Consultamos estadísticas de ejecución mediante `ai.stats`.
6. Estimamos el consumo de CU-segundos a partir de los tokens reportados.
7. Construimos un registro de auditoría y dejamos disponible su persistencia opcional.

### Consideraciones para producción

- Validar las categorías con responsables del proceso de negocio.
- Probar con diferentes tamaños de muestra antes de escalar.
- Medir tiempos, excepciones, tokens y consumo real en Capacity Metrics.
- No interpretar los CU-segundos estimados como el consumo total del notebook.
- Revisar periódicamente las tarifas oficiales, ya que pueden cambiar.

## Referencias

- [AI Functions overview](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/overview)
- [Use ai.classify with PySpark](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/classify)
- [Customize AI Functions with PySpark](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration)
- [AI Functions billing](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)
